# Recalibrate the V1 Mask R-CNN checkpoint

This notebook reuses the trained V1 weights and runs only the latest deterministic calibration and prediction stages.

In [ ]:
from pathlib import Path
import kagglehub

V1 = kagglehub.notebook_output_download(
    'jidaoluckey/solar-filament-calibrated-mask-r-cnn/versions/1',
    path='/kaggle/working/solar-v1-output',
)
checkpoints = list(Path(V1).rglob('maskrcnn-best.pt'))
assert len(checkpoints) == 1, checkpoints
CHECKPOINT = checkpoints[0]
print(CHECKPOINT)

In [ ]:
!git clone -q https://github.com/ILoveBuns/solar-filament-baseline.git /kaggle/working/solar-filament-baseline
%cd /kaggle/working/solar-filament-baseline
!python -m pip install -q -e .
!git rev-parse HEAD

In [ ]:
ROOT = '/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026'
!cp "{CHECKPOINT}" /kaggle/working/maskrcnn-best.pt
!python kaggle/train_maskrcnn.py calibrate $ROOT --checkpoint /kaggle/working/maskrcnn-best.pt
!python kaggle/train_maskrcnn.py predict $ROOT --checkpoint /kaggle/working/maskrcnn-best.pt --output /kaggle/working/submission-maskrcnn.csv

In [ ]:
from collections import Counter
import pandas as pd
from solarfil.submission import decode_mask

submission = pd.read_csv('/kaggle/working/submission-maskrcnn.csv')
assert list(submission.columns) == ['filament_id', 'segmentation_rle']
assert submission.filament_id.is_unique and submission.segmentation_rle.notna().all()
image_ids = submission.filament_id.str.rsplit('_', n=1).str[0]
test_ids = {path.stem for path in Path(ROOT, 'test/test_images').glob('*.jpeg')}
assert set(image_ids) <= test_ids
counts = Counter(image_ids)
assert max(counts.values(), default=0) <= 32
if len(submission):
    decode_mask(submission.segmentation_rle.iloc[0], (2048, 2048))
print({'rows': len(submission), 'covered_images': len(counts), 'missing_images': len(test_ids - set(image_ids)), 'max_instances': max(counts.values(), default=0)})